# Phase 1 LoRA CPT — Qwen3-4B-Base (Colab)

Continued pretrain with **Unsloth BF16 LoRA** on `Aniket200325/coder-pretrain-60gb`.

**Hardware:** prefer **A100 40GB** (~5.3 compute units/hr); stay off 80GB unless 40GB OOMs after tuning.
**Sessions:** Colab ~**12h** limit → resume from Hub/Drive `LATEST` each time.

### Must-have for speed
1. **Unsloth + packing** — logs should say `Sample packing is ACTIVE` (script fatal-exits if skipped)
2. **FA2 / xformers** when available (optional; Unsloth often enough — watch tok/s)

### Smoke (~10 min)
Small batch; confirm packing + sane tok/s before a long session.

### Full run (throughput defaults)
`--max_seq_len 2048 --per_device_train_batch_size 48 --gradient_accumulation_steps 1`  
Target **~5B tokens**. After ~10 min check `EARLY_PROJECTION`:
- peak VRAM **&lt; 30GB** → raise batch (56+)
- OOM → cut batch (40→32); keep seq 2048
- optional: trial seq 4096 only if tok/s wins

See `FINE_TUNE_DECISIONS.md` and `train_phase1.py`.


## 1. Install (Unsloth + transformers; no QLoRA)

Qwen3-4B uses **dense full attention**. Prefer Unsloth + optional `xformers` / `flash-attn`.


In [ ]:
# Unsloth Colab install — Qwen3-4B dense full-attention path
!pip install -q --upgrade pip
!pip install -q "torch" "transformers>=4.51.0" "datasets>=3.0.0" "trl" "peft" "accelerate" "huggingface_hub" "safetensors"
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Optional attention speedups (safe to fail; Unsloth often enough)
!pip install -q xformers || true

import importlib
import subprocess
import sys

import torch
import transformers

print("transformers", transformers.__version__, "torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


def _importable(name: str) -> bool:
    importlib.invalidate_caches()
    try:
        __import__(name)
        return True
    except Exception:
        return False


# flash-attn often needs a matching wheel; try quietly, do not block on failure
if not _importable("flash_attn"):
    print("Trying flash-attn (may fail on Colab — OK to continue)...")
    code = subprocess.call(
        [sys.executable, "-m", "pip", "install", "-q", "flash-attn", "--no-build-isolation"]
    )
    print("flash-attn install exit:", code)

fa = _importable("flash_attn")
xf = _importable("xformers")
print(f"Attention deps: flash_attn={fa} xformers={xf}")
print("Install OK")
if not (fa or xf):
    print("NOTE: no flash_attn/xformers yet; Unsloth kernels may still be fine — watch tok/s.")


## 2. Auth + paths (Hub token + optional Drive for resume)

In [ ]:
import os
from pathlib import Path

# --- fill these ---
HF_TOKEN = ""  # write-scoped token, or leave empty and use Colab Secrets / userdata
HUB_MODEL_ID = "YOUR_USER/coder-qwen3-4b-phase1-lora"  # private adapter repo
USE_DRIVE = True

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN", "")

assert HF_TOKEN, "Set HF_TOKEN (cell or Colab Secrets)"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

DRIVE_CKPT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_CKPT = "/content/drive/MyDrive/coder-phase1-lora"
    Path(DRIVE_CKPT).mkdir(parents=True, exist_ok=True)
    print("Drive ckpt:", DRIVE_CKPT)

OUT_DIR = Path("/content/ckpts/phase1-lora")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Local out:", OUT_DIR)

## 3. Fetch training code

Clone the GitHub repo (default). Re-run this on each new Colab session so you pick up latest `fine-tune/` scripts.

In [ ]:
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Aniket25042003/Coder.git"
REPO_DIR = Path("/content/Coder")
BRANCH = "main"  # change if you train from another branch

if REPO_DIR.exists():
    # Refresh to latest on reconnect
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", BRANCH])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH])
else:
    subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])

CODE_DIR = REPO_DIR / "fine-tune"
assert (CODE_DIR / "train_phase1.py").exists(), f"Missing train_phase1.py under {CODE_DIR}"
sys.path.insert(0, str(CODE_DIR))
print("Using", CODE_DIR)

## 4a. Live log helper + smoke test (~10 min)

Training cells stream stdout line-by-line (`PYTHONUNBUFFERED=1 python -u`) so you see `tok/s`, loss, and VRAM live — not just a final exit code.

Smoke verifies Unsloth load, packing, checkpoint write before a long session.

In [ ]:
import os
import subprocess
import sys


def run_train_live(cmd: str) -> None:
    """Stream training logs live in Colab (no buffered black hole)."""
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["TRANSFORMERS_VERBOSITY"] = "info"
    print(cmd, flush=True)
    print("--- live logs ---", flush=True)
    proc = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
        executable="/bin/bash",
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    code = proc.wait()
    print(f"--- exit {code} ---", flush=True)
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)


smoke_cmd = f"""
PYTHONUNBUFFERED=1 python -u {CODE_DIR}/train_phase1.py \
  --model unsloth/Qwen3-4B-Base \
  --dataset Aniket200325/coder-pretrain-60gb \
  --max_seq_len 2048 \
  --token_budget 1000000 \
  --max_steps 50 \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 1 \
  --save_steps 25 \
  --logging_steps 1 \
  --ckpt_minutes 5 \
  --project_after_minutes 2 \
  --output_dir {OUT_DIR}-smoke \
  --resume none \
  --no_push_to_hub
""".strip()
run_train_live(smoke_cmd)

## 4b. Full Phase 1 session (resume-aware)

- Re-run this cell on every new Colab session (`--resume auto`).
- Prefer **A100 40GB**. Defaults: **seq 2048 / batch 48 / accum 1** on `unsloth/Qwen3-4B-Base`.
- Before the ~12h kill (~11h mark), ensure a save happened (`ckpt_minutes=30` + `save_steps`).
- Startup checks: log says `Sample packing is ACTIVE` (script aborts if skipped).
- Watch live logs: `tok/s`, `VRAM` / `peak`, then **EARLY_PROJECTION** (~10 min).
- Scale: peak &lt; 30GB → try batch 56+; OOM → batch 40/32; trial seq 4096 only if tok/s improves.


In [ ]:
import time

SESSION_START = time.time()
MAX_SESSION_SEC = 11 * 3600  # push final save before typical 12h cutoff

# Throughput knobs (A100 40GB). After EARLY_PROJECTION, edit batch/seq and re-run with --resume auto.
MAX_SEQ_LEN = 2048
BATCH = 48  # try 56+ if peak VRAM < ~30GB; cut to 40/32 on OOM
ACCUM = 1

drive_arg = f"--drive_ckpt_dir {DRIVE_CKPT}" if DRIVE_CKPT else ""
hub_arg = f"--hub_model_id {HUB_MODEL_ID}" if HUB_MODEL_ID and "YOUR_USER" not in HUB_MODEL_ID else "--no_push_to_hub"

full_cmd = f"""
PYTHONUNBUFFERED=1 python -u {CODE_DIR}/train_phase1.py \
  --model unsloth/Qwen3-4B-Base \
  --dataset Aniket200325/coder-pretrain-60gb \
  --max_seq_len {MAX_SEQ_LEN} \
  --token_budget 5000000000 \
  --per_device_train_batch_size {BATCH} \
  --gradient_accumulation_steps {ACCUM} \
  --learning_rate 1e-4 \
  --logging_steps 5 \
  --save_steps 250 \
  --ckpt_minutes 30 \
  --project_after_minutes 10 \
  --remaining_colab_hours 45 \
  --output_dir {OUT_DIR} \
  --resume auto \
  {drive_arg} \
  {hub_arg}
""".strip()
print(
    f"Session soft limit: {MAX_SESSION_SEC/3600:.0f}h; timed ckpts + resume next session.",
    flush=True,
)
print(
    f"Config: seq={MAX_SEQ_LEN} batch={BATCH} accum={ACCUM} → aim ~35–38GB peak VRAM",
    flush=True,
)
run_train_live(full_cmd)

## 5. After disconnect / next session

1. Runtime → reconnect, pick **A100 40GB** again.
2. Re-run install + auth + **git clone/pull** cells.
3. Re-run **4b** only (`--resume auto` picks Drive/local `LATEST`).
4. Stop when logs show token budget reached or credits are low.

**Startup must-haves**
- `Sample packing is ACTIVE` (not “skipped”)
- Prefer `flash_attn` or `xformers` if install succeeded (not required if Unsloth tok/s is already high)

**Scaling from live logs** (edit `BATCH` / `MAX_SEQ_LEN` in 4b, then resume):
- `peak` VRAM &lt; 30GB → raise `BATCH` to 56+
- OOM → lower `BATCH` (40 → 32); keep seq 2048
- Trial `MAX_SEQ_LEN = 4096` with smaller batch only if measured `tok/s` is higher
- If packing causes NaN / exploding `grad_norm`, restart with `--no_packing` and report
- Abort on “packing skipped” / FATAL packing — do not burn a long session without packing ACTIVE

**Note:** push `fine-tune/` to GitHub `main` before recloning so Colab gets the latest scripts.
